# tinyGPT · Advanced track workspace

You write the code in `.py` files; this notebook runs the tests and the scripts.

- **Your code** is in Google Drive, at `MyDrive/tinyGPT-advanced/advanced/src/scratchgpt/`. Open
  the Files panel on the left, go to `drive/MyDrive/tinyGPT-advanced/advanced/src/scratchgpt`, and
  double-click a file to edit it. Save with Ctrl+S (Cmd+S on a Mac). Every command below
  starts a fresh Python, so it always uses your latest code.
- **The briefs** are in `advanced/projects/`, one per project. Read the brief before you
  start a project; this notebook only runs things.
- **Your checkpoints** are written to `MyDrive/tinyGPT-runs/`, so a run that is cut off can
  carry on in the next session: run the same cell again.
- **Other commands from the briefs** assume the data in `data/imdb` and runs in `runs/`. Here,
  add `data_dir={DATA}/imdb` and an `out_dir={RUNS}/...` to them, as the cells below do, and
  write run folders as `{RUNS}/small` instead of `runs/small`.

Read [Responsible use](https://github.com/cgpan/tinyGPT#responsible-use) before you share
anything your model writes.

In [ ]:
#@title Setup: run this first, in every session { display-mode: "form" }
# Keeps your code in Google Drive, so your work survives the session. You don't need to change this cell.
import os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
REPO = Path("/content/drive/MyDrive/tinyGPT-advanced")  # your copy of the code: edit it here
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/cgpan/tinyGPT.git", str(REPO)], check=True)
if not (REPO / "advanced").is_dir():
    raise FileNotFoundError(f"{REPO} exists but isn't a copy of tinyGPT. Rename or remove it, then run this cell again.")
os.chdir(REPO / "advanced")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

DATA = "/content/data"                                  # Colab's fast local disk: emptied when the session ends
SAVED = Path("/content/drive/MyDrive/tinyGPT-data")     # your prepared data, kept between sessions
RUNS = "/content/drive/MyDrive/tinyGPT-runs"            # your checkpoints, kept between sessions
if SAVED.exists() and not Path(DATA).exists():
    shutil.copytree(SAVED, DATA)
    print("Restored your prepared data from Drive.")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else
      "none. Choose Runtime > Change runtime type > GPU, then run this cell again.")

## Project 1 · The data

In [ ]:
!python -m pytest -q tests/test_01_data.py

In [ ]:
!python scripts/prepare_data.py --data {DATA}

## Project 2 · The tokenizer

In [ ]:
!python -m pytest -q tests/test_02_tokenizer.py

In [ ]:
!python scripts/train_tokenizer.py --data {DATA}/imdb --sample-mb 5

In [ ]:
#@title Keep the prepared data in Drive (after projects 1 and 2) { display-mode: "form" }
shutil.copytree(Path(DATA) / "imdb", SAVED / "imdb", dirs_exist_ok=True)
print("Saved", sorted(p.name for p in (SAVED / "imdb").iterdir()))

## Project 3 · The model

In [ ]:
!python -m pytest -q tests/test_03_model.py tests/test_rules.py

## Project 4 · Training

In [ ]:
!python -m pytest -q tests/test_04_training.py

In [ ]:
!python scripts/train.py configs/small.yaml data_dir={DATA}/imdb out_dir={RUNS}/small

In [ ]:
!python scripts/sample.py {RUNS}/small --n 3

## Project 5 · Performance

In [ ]:
!python -m pytest -q tests/test_05_performance.py

In [ ]:
!python scripts/benchmark.py configs/small.yaml configs/base.yaml data_dir={DATA}/imdb

## Project 6 · Scaling, and the final run

Write `configs/final.yaml` as the brief explains, then train it. If the session ends, run the training cell again: it resumes from the last checkpoint.

In [ ]:
!python -m pytest -q tests/test_06_scaling.py

In [ ]:
!python scripts/sweep.py configs/small.yaml data_dir={DATA}/imdb --out {RUNS}/sweep

In [ ]:
!python scripts/train.py configs/final.yaml data_dir={DATA}/imdb out_dir={RUNS}/final

## Project 7 · Evaluation

In [ ]:
!python -m pytest -q tests/test_07_evaluation.py

In [ ]:
!python scripts/evaluate.py {RUNS}/final

## Project 8 · Responsible release

In [ ]:
!python -m pytest -q tests/test_08_release.py

In [ ]:
!python scripts/watermark_check.py {RUNS}/final

In [ ]:
!python scripts/model_card.py {RUNS}/final